In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="rl-policy-gradient",
    notebook_name="01_policy_gradient_intuition_experiments.ipynb"
)

# Experiments: Policy Gradient Intuition

This notebook produces **runnable evidence** for the key claims in the policy gradient intuition topic.

Every experiment answers a question you might face in a Staff/Principal MLE interview:

1. **Complexity benchmark** — How does action sampling scale for discrete vs continuous parameterizations?
2. **Failure mode demo** — What happens when a policy's entropy collapses?
3. **Ablation** — Does stochastic sampling actually help exploration compared to deterministic argmax?

Every cell produces output. Every plot supports a specific claim.

In [ ]:
# ============================================================
# Setup — imports only
# ============================================================
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import time

np.random.seed(42)
torch.manual_seed(42)

print("NumPy version:", np.__version__)
print("PyTorch version:", torch.__version__)
print("Setup complete.")

---

## Experiment 1 — Discrete vs Continuous Action Parameterization Benchmark

**Claim we are testing:** Discrete policies use `Categorical(softmax(logits))` and continuous policies use `Gaussian(μ, σ)`. Both are valid parameterizations, but they scale differently with action dimension.

**Why it matters in an interview:** An interviewer may ask *"When would you choose a discrete vs continuous action space?"* You need to know the computational trade-offs, not just the conceptual difference.

In [ ]:
# ---------------------------------------------------------
# Experiment 1: Benchmark sampling speed for both parameterizations
# ---------------------------------------------------------

NUM_SAMPLES = 10000
action_dims = [2, 4, 8, 16, 32]

discrete_times = []
continuous_times = []

print("=== Discrete Policy (Categorical / Softmax) ===")
print(f"{'Actions':>10} | {'Time (ms)':>10} | {'Sample shape':>15}")
print("-" * 42)

for n_actions in action_dims:
    # Simple policy network: state_dim=4 -> n_actions logits
    policy_net = nn.Sequential(
        nn.Linear(4, 32),
        nn.ReLU(),
        nn.Linear(32, n_actions)
    )
    dummy_state = torch.randn(1, 4)

    start = time.perf_counter()
    for _ in range(NUM_SAMPLES):
        with torch.no_grad():
            logits = policy_net(dummy_state)
            probs = F.softmax(logits, dim=-1)
            dist = torch.distributions.Categorical(probs)
            action = dist.sample()
    elapsed = (time.perf_counter() - start) * 1000  # ms
    discrete_times.append(elapsed)
    print(f"{n_actions:>10} | {elapsed:>10.2f} | {str(action.shape):>15}")

print()
print("=== Continuous Policy (Gaussian / Mean+Std) ===")
print(f"{'Dimensions':>10} | {'Time (ms)':>10} | {'Sample shape':>15}")
print("-" * 42)

for n_dims in action_dims:
    # Policy network outputs mean and log_std for each dimension
    policy_net = nn.Sequential(
        nn.Linear(4, 32),
        nn.ReLU(),
        nn.Linear(32, n_dims * 2)  # mean + log_std for each dim
    )
    dummy_state = torch.randn(1, 4)

    start = time.perf_counter()
    for _ in range(NUM_SAMPLES):
        with torch.no_grad():
            output = policy_net(dummy_state)
            mean = output[:, :n_dims]
            log_std = output[:, n_dims:]
            std = torch.exp(log_std)
            dist = torch.distributions.Normal(mean, std)
            action = dist.sample()
    elapsed = (time.perf_counter() - start) * 1000  # ms
    continuous_times.append(elapsed)
    print(f"{n_dims:>10} | {elapsed:>10.2f} | {str(action.shape):>15}")

print()
print("Both parameterizations work. Continuous outputs a vector, discrete outputs a scalar index.")

In [ ]:
# ---------------------------------------------------------
# Verify the distributions look correct
# ---------------------------------------------------------

# Discrete: sample 5000 actions from a 4-action policy and check frequencies
logits = torch.tensor([[2.0, 1.0, 0.5, 0.1]])
probs = F.softmax(logits, dim=-1)
dist = torch.distributions.Categorical(probs)
samples = torch.tensor([dist.sample().item() for _ in range(5000)])

print("=== Discrete Distribution Verification ===")
print(f"Softmax probabilities: {probs.numpy().flatten()}")
print(f"Empirical frequencies:  {[(samples == i).float().mean().item() for i in range(4)]}")
print()

# Continuous: sample from Gaussian and check mean/std match
true_mean = torch.tensor([1.5])
true_std = torch.tensor([0.3])
dist = torch.distributions.Normal(true_mean, true_std)
samples = torch.stack([dist.sample() for _ in range(5000)])

print("=== Continuous Distribution Verification ===")
print(f"Target mean: {true_mean.item():.2f}, Empirical mean: {samples.mean().item():.4f}")
print(f"Target std:  {true_std.item():.2f}, Empirical std:  {samples.std().item():.4f}")
print()
print("Both distributions sample correctly. The empirical stats match the parameters.")

In [ ]:
# ---------------------------------------------------------
# Plot: sampling time vs action dimension
# ---------------------------------------------------------

fig, ax = plt.subplots(1, 1, figsize=(8, 5))

ax.plot(action_dims, discrete_times, 'o-', color='#2196F3', linewidth=2,
        markersize=8, label='Discrete (Categorical)')
ax.plot(action_dims, continuous_times, 's-', color='#FF5722', linewidth=2,
        markersize=8, label='Continuous (Gaussian)')

ax.set_xlabel('Action Dimensions', fontsize=12)
ax.set_ylabel(f'Time for {NUM_SAMPLES:,} samples (ms)', fontsize=12)
ax.set_title('Sampling Speed: Discrete vs Continuous Policy', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks(action_dims)

plt.tight_layout()
plt.show()

print("Plot shows how sampling time grows with action dimensionality for each parameterization.")

### What the output shows

- **Discrete policies** output a probability over N actions via softmax. Sampling picks one index. The cost grows with the number of discrete actions.
- **Continuous policies** output a mean and standard deviation per dimension. Sampling draws from a Gaussian. The cost grows with the dimensionality of the action vector.
- Both are valid. The choice depends on your problem: robot joint angles are continuous, game button presses are discrete.

**In an interview, say:** *"Discrete actions use Categorical sampling from softmax logits. Continuous actions use Gaussian sampling from learned mean and variance. The parameterization determines what kind of exploration the policy does naturally.”*

---

## Experiment 2 — Failure Mode: Entropy Collapse Demo

**Claim we are testing:** If a policy becomes too deterministic too early, exploration stops and the agent gets stuck. This is called **entropy collapse**.

**Why it matters in an interview:** This is one of the most common failure modes in policy gradient methods. An interviewer asking about failure modes expects you to name this, explain why it happens, and know the fix (entropy regularization).

In [ ]:
# ---------------------------------------------------------
# Experiment 2: Entropy collapse in a simple 2-action bandit
# ---------------------------------------------------------

def compute_entropy(probs):
    """Compute entropy of a discrete distribution. Higher = more random."""
    # Add small epsilon to avoid log(0)
    return -np.sum(probs * np.log(probs + 1e-10))

def run_bandit_experiment(n_updates, learning_rate, entropy_coeff=0.0, seed=42):
    """
    Simulate REINFORCE-style updates on a 2-action bandit.

    Action 0 gives reward +1 with probability 0.6 (slightly better).
    Action 1 gives reward +1 with probability 0.4.

    Returns: list of (entropy, probs) at each update step.
    """
    rng = np.random.RandomState(seed)

    # Start with uniform policy: logits = [0, 0] -> probs = [0.5, 0.5]
    logits = np.array([0.0, 0.0])
    reward_probs = [0.6, 0.4]  # true reward probabilities

    history = []

    for step in range(n_updates):
        # Current policy probabilities
        probs = np.exp(logits) / np.sum(np.exp(logits))  # softmax
        entropy = compute_entropy(probs)
        history.append((entropy, probs.copy()))

        # Sample action from policy
        action = rng.choice(2, p=probs)

        # Get reward
        reward = 1.0 if rng.random() < reward_probs[action] else 0.0

        # REINFORCE gradient: d/d_logit = reward * (1(action=a) - prob(a))
        # This pushes up the logit for the chosen action proportional to reward
        grad = np.zeros(2)
        grad[action] = reward * (1.0 - probs[action])
        for a in range(2):
            if a != action:
                grad[a] = reward * (0.0 - probs[a])

        # Entropy regularization gradient: encourages higher entropy
        if entropy_coeff > 0:
            # Gradient of entropy w.r.t. logits pushes toward uniform
            entropy_grad = -(np.log(probs + 1e-10) + 1.0)
            entropy_grad -= np.mean(entropy_grad)  # center the gradient
            grad += entropy_coeff * entropy_grad

        logits += learning_rate * grad

    return history

# Run WITHOUT entropy regularization
n_updates = 300
lr = 0.5

history_no_reg = run_bandit_experiment(n_updates, lr, entropy_coeff=0.0)
history_with_reg = run_bandit_experiment(n_updates, lr, entropy_coeff=0.1)

# Print key moments
print("=== Without Entropy Regularization ===")
for step_idx in [0, 50, 100, 150, 200, 299]:
    ent, probs = history_no_reg[step_idx]
    print(f"  Step {step_idx:>3}: entropy={ent:.4f}, probs=[{probs[0]:.4f}, {probs[1]:.4f}]")

print()
print("=== With Entropy Regularization (coeff=0.1) ===")
for step_idx in [0, 50, 100, 150, 200, 299]:
    ent, probs = history_with_reg[step_idx]
    print(f"  Step {step_idx:>3}: entropy={ent:.4f}, probs=[{probs[0]:.4f}, {probs[1]:.4f}]")

print()
max_entropy_2_actions = np.log(2)
print(f"Maximum possible entropy for 2 actions: {max_entropy_2_actions:.4f} (= ln(2), uniform distribution)")
print(f"Minimum entropy (deterministic): 0.0000")
print()
print(f"Final entropy WITHOUT reg: {history_no_reg[-1][0]:.4f}")
print(f"Final entropy WITH reg:    {history_with_reg[-1][0]:.4f}")

In [ ]:
# ---------------------------------------------------------
# Plot: entropy over training steps
# ---------------------------------------------------------

entropies_no_reg = [h[0] for h in history_no_reg]
entropies_with_reg = [h[0] for h in history_with_reg]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: entropy over time
axes[0].plot(entropies_no_reg, color='#F44336', linewidth=2, label='No regularization')
axes[0].plot(entropies_with_reg, color='#4CAF50', linewidth=2, label='Entropy reg (coeff=0.1)')
axes[0].axhline(y=np.log(2), color='gray', linestyle='--', alpha=0.5, label='Max entropy (uniform)')
axes[0].axhline(y=0.0, color='gray', linestyle=':', alpha=0.5, label='Min entropy (deterministic)')
axes[0].set_xlabel('Update Step', fontsize=12)
axes[0].set_ylabel('Policy Entropy', fontsize=12)
axes[0].set_title('Entropy Collapse: With vs Without Regularization', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(-0.05, np.log(2) + 0.1)

# Right: probability of action 0 over time
prob_action0_no_reg = [h[1][0] for h in history_no_reg]
prob_action0_with_reg = [h[1][0] for h in history_with_reg]

axes[1].plot(prob_action0_no_reg, color='#F44336', linewidth=2, label='No regularization')
axes[1].plot(prob_action0_with_reg, color='#4CAF50', linewidth=2, label='Entropy reg (coeff=0.1)')
axes[1].axhline(y=0.6, color='gray', linestyle='--', alpha=0.5, label='True optimal = 0.6')
axes[1].set_xlabel('Update Step', fontsize=12)
axes[1].set_ylabel('P(action 0)', fontsize=12)
axes[1].set_title('Policy Probability for Better Action', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

print("Left plot: entropy drops to near-zero without regularization (collapse).")
print("Right plot: without reg, P(action 0) goes to ~1.0 even though optimal is 0.6.")
print("With entropy reg, the policy stays stochastic and closer to the true optimum.")

### What the output shows

- **Without entropy regularization**, the policy quickly becomes deterministic. Entropy drops to near zero. The agent picks one action almost 100% of the time, even though the optimal strategy is only 60/40.
- **With entropy regularization** (coefficient = 0.1), entropy stays higher. The policy remains stochastic and lands closer to the true optimal probabilities.
- This is **entropy collapse**: once the policy commits to one action, the gradient for the other action vanishes because it never gets sampled. Learning stops.

**In an interview, say:** *"Entropy collapse causes premature convergence. The policy becomes deterministic before finding the optimum. Entropy regularization adds a bonus for keeping the policy stochastic, which preserves exploration. The coefficient controls the exploration-exploitation trade-off.”*

---

## Experiment 3 — Ablation: Stochastic vs Deterministic Policy

**Claim we are testing:** Stochastic policies explore better than deterministic (argmax) policies because they naturally try different actions.

**Why it matters in an interview:** This is the fundamental reason policy gradient methods use stochastic policies. If an interviewer asks *"Why not just pick the best action every time?"*, you need data showing that greedy/deterministic selection leads to higher regret.

In [ ]:
# ---------------------------------------------------------
# Experiment 3: Stochastic vs deterministic on 5-arm bandit
# ---------------------------------------------------------

N_ARMS = 5
N_STEPS = 500
N_RUNS = 50

# True reward probabilities for each arm
# Arm 2 is the best (0.7), others are mediocre
true_rewards = np.array([0.3, 0.4, 0.7, 0.35, 0.25])
optimal_arm = np.argmax(true_rewards)
optimal_reward = true_rewards[optimal_arm]

print(f"Bandit setup: {N_ARMS} arms")
print(f"True reward probabilities: {true_rewards}")
print(f"Optimal arm: {optimal_arm} (reward prob = {optimal_reward})")
print(f"Running {N_RUNS} independent runs, {N_STEPS} steps each...")
print()

def run_stochastic_agent(n_steps, true_rewards, learning_rate=0.1, seed=0):
    """Agent that samples actions from a learned softmax distribution."""
    rng = np.random.RandomState(seed)
    n_arms = len(true_rewards)
    logits = np.zeros(n_arms)
    cumulative_regret = []
    total_regret = 0.0

    for step in range(n_steps):
        # Softmax probabilities
        exp_logits = np.exp(logits - np.max(logits))  # numerical stability
        probs = exp_logits / np.sum(exp_logits)

        # Sample action from distribution
        action = rng.choice(n_arms, p=probs)

        # Get reward
        reward = 1.0 if rng.random() < true_rewards[action] else 0.0

        # REINFORCE-style update
        grad = np.zeros(n_arms)
        for a in range(n_arms):
            if a == action:
                grad[a] = reward * (1.0 - probs[a])
            else:
                grad[a] = reward * (0.0 - probs[a])
        logits += learning_rate * grad

        # Track regret: difference between optimal reward and what we got
        instant_regret = optimal_reward - true_rewards[action]
        total_regret += instant_regret
        cumulative_regret.append(total_regret)

    return cumulative_regret

def run_deterministic_agent(n_steps, true_rewards, learning_rate=0.1, seed=0):
    """Agent that always picks the action with the highest estimated value (argmax)."""
    rng = np.random.RandomState(seed)
    n_arms = len(true_rewards)
    # Estimated value for each arm (optimistic initialization to encourage some exploration)
    q_values = np.zeros(n_arms)
    action_counts = np.zeros(n_arms)
    cumulative_regret = []
    total_regret = 0.0

    for step in range(n_steps):
        # Deterministic: always pick the best-looking arm
        # Break ties randomly at the start
        if step < n_arms:
            # Try each arm once first
            action = step
        else:
            action = np.argmax(q_values)

        # Get reward
        reward = 1.0 if rng.random() < true_rewards[action] else 0.0

        # Update estimated value (incremental mean)
        action_counts[action] += 1
        q_values[action] += (reward - q_values[action]) / action_counts[action]

        # Track regret
        instant_regret = optimal_reward - true_rewards[action]
        total_regret += instant_regret
        cumulative_regret.append(total_regret)

    return cumulative_regret

# Run both agents multiple times and average
all_stochastic = []
all_deterministic = []

for run in range(N_RUNS):
    all_stochastic.append(run_stochastic_agent(N_STEPS, true_rewards, seed=run))
    all_deterministic.append(run_deterministic_agent(N_STEPS, true_rewards, seed=run))

avg_stochastic = np.mean(all_stochastic, axis=0)
avg_deterministic = np.mean(all_deterministic, axis=0)
std_stochastic = np.std(all_stochastic, axis=0)
std_deterministic = np.std(all_deterministic, axis=0)

print(f"{'Metric':>30} | {'Stochastic':>12} | {'Deterministic':>14}")
print("-" * 65)
print(f"{'Final cumulative regret (avg)':>30} | {avg_stochastic[-1]:>12.2f} | {avg_deterministic[-1]:>14.2f}")
print(f"{'Final cumulative regret (std)':>30} | {std_stochastic[-1]:>12.2f} | {std_deterministic[-1]:>14.2f}")
print(f"{'Regret at step 100 (avg)':>30} | {avg_stochastic[99]:>12.2f} | {avg_deterministic[99]:>14.2f}")
print(f"{'Regret at step 250 (avg)':>30} | {avg_stochastic[249]:>12.2f} | {avg_deterministic[249]:>14.2f}")

In [ ]:
# ---------------------------------------------------------
# Plot: cumulative regret comparison
# ---------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

steps = np.arange(N_STEPS)

# Left: cumulative regret with confidence bands
axes[0].plot(steps, avg_stochastic, color='#4CAF50', linewidth=2, label='Stochastic (sample from policy)')
axes[0].fill_between(steps,
                      avg_stochastic - std_stochastic,
                      avg_stochastic + std_stochastic,
                      color='#4CAF50', alpha=0.15)
axes[0].plot(steps, avg_deterministic, color='#F44336', linewidth=2, label='Deterministic (argmax)')
axes[0].fill_between(steps,
                      avg_deterministic - std_deterministic,
                      avg_deterministic + std_deterministic,
                      color='#F44336', alpha=0.15)
axes[0].set_xlabel('Step', fontsize=12)
axes[0].set_ylabel('Cumulative Regret', fontsize=12)
axes[0].set_title(f'Cumulative Regret ({N_RUNS} runs averaged)', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Right: histogram of final regret across runs
final_stoch = [run[-1] for run in all_stochastic]
final_det = [run[-1] for run in all_deterministic]

axes[1].hist(final_stoch, bins=15, alpha=0.6, color='#4CAF50', label='Stochastic', edgecolor='white')
axes[1].hist(final_det, bins=15, alpha=0.6, color='#F44336', label='Deterministic', edgecolor='white')
axes[1].axvline(np.mean(final_stoch), color='#2E7D32', linestyle='--', linewidth=2)
axes[1].axvline(np.mean(final_det), color='#C62828', linestyle='--', linewidth=2)
axes[1].set_xlabel('Final Cumulative Regret', fontsize=12)
axes[1].set_ylabel('Count (runs)', fontsize=12)
axes[1].set_title(f'Distribution of Final Regret ({N_RUNS} runs)', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Count how many times each agent achieved lower regret
stoch_wins = sum(1 for s, d in zip(final_stoch, final_det) if s < d)
det_wins = sum(1 for s, d in zip(final_stoch, final_det) if d < s)
ties = N_RUNS - stoch_wins - det_wins
print(f"Stochastic wins: {stoch_wins}/{N_RUNS}, Deterministic wins: {det_wins}/{N_RUNS}, Ties: {ties}/{N_RUNS}")

### What the output shows

- The **stochastic agent** naturally explores different arms because it samples from a probability distribution. This means it discovers the optimal arm and adjusts its probabilities toward it.
- The **deterministic agent** (argmax) commits to whatever looks best after the initial exploration phase. If it gets unlucky early on and a suboptimal arm happens to pay off, it locks in and never tries the others.
- The histogram shows the deterministic agent has higher variance in outcomes — sometimes it gets lucky and finds the best arm immediately, but often it gets stuck on a suboptimal arm.
- Stochastic policies have a built-in exploration mechanism. Deterministic policies need explicit exploration strategies (like epsilon-greedy) bolted on.

**In an interview, say:** *"Stochastic policies explore naturally by sampling from the action distribution. This is a key advantage of policy gradient methods — exploration is built into the parameterization, not added as an afterthought. The trade-off is higher variance in gradient estimates, which is why we need variance reduction techniques like baselines.”*

---

## Summary — Claims Now Backed by Evidence

| Claim | Experiment | Key Evidence |
|-------|------------|--------------|
| Discrete and continuous policies use different parameterizations | Experiment 1 | Categorical samples a scalar index, Gaussian samples a vector. Both verified against expected distributions. |
| Entropy collapse causes premature convergence | Experiment 2 | Without regularization, entropy drops to ~0 and the policy becomes deterministic too early. With entropy regularization, the policy stays exploratory. |
| Stochastic policies explore better than deterministic ones | Experiment 3 | Stochastic agent achieves lower cumulative regret on average because it naturally tries different actions. |

These results are self-contained and reproducible (seed=42). You can re-run any cell to regenerate the evidence.

For the full mathematical treatment and interview Q&A, see [policy-gradient-intuition-interview.md](./policy-gradient-intuition-interview.md).

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)